# Mascarade Fine-Tuning on Kaggle (Free 2x T4)

Fine-tune small LLMs using Kaggle's free GPU quota (30h/week, 2x T4 16GB).

**Domains**: KiCad/EDA, STM32/Embedded, Generic Code

## Setup
1. Upload this notebook to Kaggle
2. Upload your datasets from `./datasets/` as a Kaggle Dataset
3. Enable GPU accelerator (Settings > Accelerator > GPU T4 x2)
4. Run all cells

In [ ]:
# Install dependencies
!pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.12.0 \
    bitsandbytes==0.43.2 datasets trl unsloth

In [ ]:
import torch
import os

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"GPUs available: {torch.cuda.device_count()}")
print(f"PyTorch: {torch.__version__}")

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# ============================================================
# CONFIGURATION - Modify this cell
# ============================================================

DOMAIN = "stm32"  # "kicad" | "stm32" | "generic"

# Dataset path (adjust to your Kaggle dataset mount)
# After uploading datasets/ as a Kaggle Dataset named "mascarade-datasets":
DATASET_PATH = f"/kaggle/input/mascarade-datasets/{DOMAIN}_dataset.txt"

# Model selection - T4 16GB can handle 3B with QLoRA
MODELS = {
    "kicad": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "stm32": "bigcode/starcoderbase-1b",
    "generic": "microsoft/phi-2",
}
MODEL_NAME = MODELS[DOMAIN]

# Training hyperparameters
MAX_SEQ_LENGTH = 512       # T4 can handle 512 easily
BATCH_SIZE = 4             # T4 16GB allows batch>1
GRADIENT_ACCUM = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
LORA_R = 16                # Higher rank = better quality
LORA_ALPHA = 32

OUTPUT_DIR = f"/kaggle/working/fine_tuned_{DOMAIN}"

print(f"Domain: {DOMAIN}")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH}")

In [ ]:
# ============================================================
# Load model with 4-bit quantization (QLoRA)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# ============================================================
# Configure LoRA
# ============================================================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ============================================================
# Prepare dataset
# ============================================================
from datasets import load_dataset

dataset = load_dataset('text', data_files={'train': DATASET_PATH})
split = dataset['train'].train_test_split(test_size=0.1, seed=42)

def tokenize(examples):
    out = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='max_length',
    )
    out['labels'] = out['input_ids'].copy()
    return out

train_ds = split['train'].map(tokenize, batched=True, remove_columns=['text'])
eval_ds = split['test'].map(tokenize, batched=True, remove_columns=['text'])

print(f"Train: {len(train_ds)} samples, Eval: {len(eval_ds)} samples")

In [ ]:
# ============================================================
# Train
# ============================================================
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting training...")
result = trainer.train()
print(f"Training done. Loss: {result.metrics['train_loss']:.4f}")

In [ ]:
# ============================================================
# Save model (LoRA adapters only, small files)
# ============================================================
save_path = f"{OUTPUT_DIR}/final"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# List saved files
import os
for f in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, f)) / 1024**2
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# ============================================================
# Test inference
# ============================================================
TEST_PROMPTS = {
    "kicad": "Generate a KiCad schematic for a simple LED circuit with resistor",
    "stm32": "Write STM32 HAL code for UART transmission at 115200 baud",
    "generic": "Write a Python function to implement binary search",
}

prompt = TEST_PROMPTS[DOMAIN]
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nGenerated:\n{result}")

In [ ]:
# ============================================================
# Download: zip and save as Kaggle output
# ============================================================
import shutil
zip_path = f"/kaggle/working/fine_tuned_{DOMAIN}"
shutil.make_archive(zip_path, 'zip', save_path)
print(f"Zipped to {zip_path}.zip")
print("Go to Output tab to download the zip file.")